# L6: Multi-agent Collaboration for Financial Analysis

In this lesson, you will learn ways for making agents collaborate with each other.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [16]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import libraries, APIs and LLM

In [17]:
from crewai import Agent, Task, Crew

**Note**: 
- The video uses `gpt-4-turbo`, but due to certain constraints, and in order to offer this course for free to everyone, the code you'll run here will use `gpt-3.5-turbo`.
- You can use `gpt-4-turbo` when you run the notebook _locally_ (using `gpt-4-turbo` will not work on the platform)
- Thank you for your understanding!

In [18]:
import os
from utils import get_openai_api_key, get_serper_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["SERPER_API_KEY"] = get_serper_api_key()

## crewAI Tools

In [19]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

## Creating Agents

In [20]:
data_analyst_agent = Agent(
    role="Data Analyst",
    goal="Monitor and analyze market data in real-time "
         "to identify trends and predict market movements.",
    backstory="Specializing in financial markets, this agent "
              "uses statistical modeling and machine learning "
              "to provide crucial insights. With a knack for data, "
              "the Data Analyst Agent is the cornerstone for "
              "informing trading decisions.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [21]:
trading_strategy_agent = Agent(
    role="Trading Strategy Developer",
    goal="Develop and test various trading strategies based "
         "on insights from the Data Analyst Agent.",
    backstory="Equipped with a deep understanding of financial "
              "markets and quantitative analysis, this agent "
              "devises and refines trading strategies. It evaluates "
              "the performance of different approaches to determine "
              "the most profitable and risk-averse options.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [22]:
execution_agent = Agent(
    role="Trade Advisor",
    goal="Suggest optimal trade execution strategies "
         "based on approved trading strategies.",
    backstory="This agent specializes in analyzing the timing, price, "
              "and logistical details of potential trades. By evaluating "
              "these factors, it provides well-founded suggestions for "
              "when and how trades should be executed to maximize "
              "efficiency and adherence to strategy.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [23]:
risk_management_agent = Agent(
    role="Risk Advisor",
    goal="Evaluate and provide insights on the risks "
         "associated with potential trading activities.",
    backstory="Armed with a deep understanding of risk assessment models "
              "and market dynamics, this agent scrutinizes the potential "
              "risks of proposed trades. It offers a detailed analysis of "
              "risk exposure and suggests safeguards to ensure that "
              "trading activities align with the firm’s risk tolerance.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

## Creating Tasks

In [24]:
# Task for Data Analyst Agent: Analyze Market Data
data_analysis_task = Task(
    description=(
        "Continuously monitor and analyze market data for "
        "the selected stock ({stock_selection}). "
        "Use statistical modeling and machine learning to "
        "identify trends and predict market movements."
    ),
    expected_output=(
        "Insights and alerts about significant market "
        "opportunities or threats for {stock_selection}."
    ),
    agent=data_analyst_agent,
)

In [25]:
# Task for Trading Strategy Agent: Develop Trading Strategies
strategy_development_task = Task(
    description=(
        "Develop and refine trading strategies based on "
        "the insights from the Data Analyst and "
        "user-defined risk tolerance ({risk_tolerance}). "
        "Consider trading preferences ({trading_strategy_preference})."
    ),
    expected_output=(
        "A set of potential trading strategies for {stock_selection} "
        "that align with the user's risk tolerance."
    ),
    agent=trading_strategy_agent,
)


In [26]:
# Task for Trade Advisor Agent: Plan Trade Execution
execution_planning_task = Task(
    description=(
        "Analyze approved trading strategies to determine the "
        "best execution methods for {stock_selection}, "
        "considering current market conditions and optimal pricing."
    ),
    expected_output=(
        "Detailed execution plans suggesting how and when to "
        "execute trades for {stock_selection}."
    ),
    agent=execution_agent,
)


In [27]:
# Task for Risk Advisor Agent: Assess Trading Risks
risk_assessment_task = Task(
    description=(
        "Evaluate the risks associated with the proposed trading "
        "strategies and execution plans for {stock_selection}. "
        "Provide a detailed analysis of potential risks "
        "and suggest mitigation strategies."
    ),
    expected_output=(
        "A comprehensive risk analysis report detailing potential "
        "risks and mitigation recommendations for {stock_selection}."
    ),
    agent=risk_management_agent,
)

## Creating the Crew
- The `Process` class helps to delegate the workflow to the Agents (kind of like a Manager at work)
- In the example below, it will run this hierarchically.
- `manager_llm` lets you choose the "manager" LLM you want to use.

In [28]:
from crewai import Crew, Process
from langchain_openai import ChatOpenAI

# Define the crew with agents and tasks
financial_trading_crew = Crew(
    agents=[data_analyst_agent, 
            trading_strategy_agent, 
            execution_agent, 
            risk_management_agent],
    
    tasks=[data_analysis_task, 
           strategy_development_task, 
           execution_planning_task, 
           risk_assessment_task],
    
    manager_llm=ChatOpenAI(model="gpt-3.5-turbo", 
                           temperature=0.7),
    process=Process.hierarchical,
    verbose=True
)

2025-06-13 12:20:50,754 - 140320813366144 - __init__.py-__init__:518 - WARNING: Overriding of current TracerProvider is not allowed


## Running the Crew

- Set the inputs for the execution of the crew.

In [29]:
# Example data for kicking off the process
financial_trading_inputs = {
    'stock_selection': 'AAPL',
    'initial_capital': '100000',
    'risk_tolerance': 'Medium',
    'trading_strategy_preference': 'Day Trading',
    'news_impact_consideration': True
}

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [30]:
### this execution will take some time to run
result = financial_trading_crew.kickoff(inputs=financial_trading_inputs)

 [DEBUG]: Working Agent: Crew Manager
 [INFO]: Starting Task: Continuously monitor and analyze market data for the selected stock (AAPL). Use statistical modeling and machine learning to identify trends and predict market movements.


> Entering new CrewAgentExecutor chain...
I need to delegate the task of continuously monitoring and analyzing market data for AAPL to the Data Analyst to ensure we are able to identify trends and predict market movements effectively.

Action: Delegate work to co-worker
Action Input: {
    "coworker": "Data Analyst",
    "task": "Continuously monitor and analyze market data for AAPL",
    "context": "The task involves using statistical modeling and machine learning to identify trends and predict market movements for AAPL. This requires a deep understanding of market data and the ability to generate insights and alerts about significant market opportunities or threats."
}


> Entering new CrewAgentExecutor chain...
I need to gather real-time market data fo

Final Answer: The current market data for AAPL shows that the stock closed at $199.64, with a slight increase of 0.22%. The stock's performance for different time frames includes a 1-day decrease of 0.71%, a 1-month decrease of 5.50%, a 6-month decrease of 19.66%, a year-to-date decrease of 20.45%, and a 1-year decrease of 6.51%. Other key metrics such as the market cap, PE ratio, EPS, and forward dividend yield are also provided. The company overview, financial highlights, and research analysis are detailed, providing a comprehensive view of Apple Inc.'s current position in the market.

> Finished chain.
 

The current market data for AAPL shows that the stock closed at $199.64, with a slight increase of 0.22%. The stock's performance for different time frames includes a 1-day decrease of 0.71%, a 1-month decrease of 5.50%, a 6-month decrease of 19.66%, a year-to-date decrease of 20.45%, and a 1-year decrease of 6.51%. Other key metrics such as the market cap, PE ratio, EPS, and forwa

Final Answer:
After analyzing the market data and insights provided by the Data Analyst, here are the potential trading strategies for AAPL that align with the user's risk tolerance:

1. Long-Term Investment Strategy: Considering the downward trend in AAPL's stock price over various time frames, a long-term investment strategy could be beneficial. This strategy involves buying and holding AAPL stock for an extended period, taking advantage of potential growth opportunities in the future.

2. Options Trading Strategy: Given the neutral performance forecast for AAPL over the next 12 months, an options trading strategy could be implemented. This strategy involves buying call options if bullish on AAPL's performance or buying put options if bearish, providing flexibility based on market conditions.

3. Day Trading Strategy: For investors with a preference for day trading, a strategy could be developed based on intraday price movements of AAPL stock. This strategy involves buying and sellin

I need to gather information on the risks associated with the proposed trading strategies and execution plans for AAPL in order to provide a detailed analysis and suggest mitigation strategies.

Action: Delegate work to co-worker
Action Input: {"coworker": "Risk Advisor", "task": "Evaluate risks associated with proposed trading strategies and execution plans for AAPL", "context": "I was unable to access the website content for trading strategies for AAPL. Please provide a comprehensive risk analysis report detailing potential risks and mitigation recommendations."}

> Entering new CrewAgentExecutor chain...
I need to gather information on trading strategies for AAPL in order to evaluate the risks associated with them and provide mitigation recommendations. Since I couldn't access the website content directly, I should search the internet for relevant information.

Action: Search the internet
Action Input: {"search_query": "AAPL trading strategies and risks"} 


Search results: Title: A

Final Answer:

Comprehensive Risk Analysis Report for AAPL Trading Strategies:

Upon evaluating the risks associated with the proposed trading strategies and execution plans for AAPL, the following potential risks have been identified:

1. High Implied Volatility: The options strategy of selling a short strangle in Apple stock comes with high premiums, indicating potential high risks due to increased implied volatility.

2. Fundamental and Technical Risks: Traders should consider company fundamentals, stock technical characteristics, and investor risk tolerance when devising trading strategies for AAPL to mitigate potential risks associated with stock trading.

3. Overlooking Risks: While Apple has experienced strong stock appreciation and growth, there may be risks that investors are overlooking. It is crucial to be aware of potential downsides to make informed trading decisions.

4. Legal Challenges: Legal risks, such as antitrust issues, pose a threat to Apple's earnings, with aroun

- Display the final result as Markdown.

In [31]:
from IPython.display import Markdown
Markdown(result)

Comprehensive Risk Analysis Report for AAPL Trading Strategies:

Upon evaluating the risks associated with the proposed trading strategies and execution plans for AAPL, the following potential risks have been identified:

1. High Implied Volatility: The options strategy of selling a short strangle in Apple stock comes with high premiums, indicating potential high risks due to increased implied volatility.

2. Fundamental and Technical Risks: Traders should consider company fundamentals, stock technical characteristics, and investor risk tolerance when devising trading strategies for AAPL to mitigate potential risks associated with stock trading.

3. Overlooking Risks: While Apple has experienced strong stock appreciation and growth, there may be risks that investors are overlooking. It is crucial to be aware of potential downsides to make informed trading decisions.

4. Legal Challenges: Legal risks, such as antitrust issues, pose a threat to Apple's earnings, with around 20% of earnings potentially at risk if legal challenges do not go in Apple's favor.

5. Regulatory Pressures and Supply Chain Challenges: Apple faces risks related to regulatory pressures, supply chain challenges, and slow innovation, which could impact the company's performance and trading outcomes.

To mitigate these risks associated with trading AAPL, traders are advised to conduct thorough research, diversify their portfolio, and consider risk management strategies to protect against potential losses. It is essential to stay informed about market trends, company developments, and external factors that could influence AAPL's trading performance.